# Chapter 5 &mdash; Exponential Blow-Up: "Third-Last Bit is a 1"

**Concept 10 of the Chapter 5 decomposition:** *Exponential Blow-Up: "Third-Last Bit is a 1"*

Look-back languages force the machine to remember every recent bit pattern &mdash; 8 states here, $2^N$ in general.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Exponential-Blow-Up/Concept-Exponential-Blow-Up.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


"The **third-last** symbol is a 1" sounds tiny. But a DFA reads left to right and
cannot know where the end is, so at every moment it must remember **the last three
bits exactly** &mdash; all $2^3$ patterns.

That is the shape of the blow-up: look-back of $N$ needs $2^N$ states, and the minimal
DFA really does have that many (Concept 11 proves it).

This is the first place where a language that is trivial to *describe* is expensive to
*recognise deterministically*. An NFA (Chapter 7) needs only $N{+}1$ states.

## 2. Definitions

### The specification

In [ ]:
def in_L(s): return len(s) >= 3 and s[-3] == '1'

### The DFA: state = the last three bits

In [ ]:
import itertools
def build_thirdlast():
    lines = ['DFA']
    windows = [''.join(p) for p in itertools.product('01', repeat=3)]
    # start states for the first three symbols
    lines += ['I    : 0 -> S_0', 'I    : 1 -> S_1',
              'S_0  : 0 -> S_00', 'S_0  : 1 -> S_01',
              'S_1  : 0 -> S_10', 'S_1  : 1 -> S_11']
    for w in ['00','01','10','11']:
        for b in '01':
            nxt = (w + b)
            pre = 'F_' if nxt[0] == '1' else 'S_'
            lines.append('S_%s : %s -> %s%s' % (w, b, pre, nxt))
    for w in windows:
        for b in '01':
            nxt = (w + b)[1:]
            pre = 'F_' if nxt[0] == '1' else 'S_'
            src = ('F_' if w[0] == '1' else 'S_') + w
            lines.append('%s : %s -> %s%s' % (src, b, pre, nxt))
    return md2mc('\n'.join(lines))

T3 = build_thirdlast()

## 3. Tests

Eight window states plus the three warm-up states.

In [ ]:
print("|Q| =", len(T3["Q"]))
print("minimized  :", len(min_dfa(T3)["Q"]))

It recognises the language.

In [ ]:
from itertools import product
bad = [''.join(p) for k in range(11) for p in product('01', repeat=k)
       if accepts_dfa(T3, ''.join(p)) != in_L(''.join(p))]
print("mismatches up to length 10 :", len(bad))
assert not bad
for s in ['100', '1000', '0100', '011', '111', '00']:
    print("  %-6r third-last is 1? %s" % (s, accepts_dfa(T3, s)))

The cost grows as $2^N$: here is the minimal size for $N=1,2,3,4$.

In [ ]:
def nth_last(N):
    lines = ['DFA']
    wins = [''.join(p) for p in product('01', repeat=N)]
    def nm(w): return ('F_' if w[0] == '1' else 'S_') + w
    for k in range(N):                       # warm-up
        for p in product('01', repeat=k):
            src = 'I' if k == 0 else 'S_' + ''.join(p)
            for b in '01':
                t = ''.join(p) + b
                lines.append('%s : %s -> %s' % (src, b, nm(t) if len(t) == N else 'S_' + t))
    for w in wins:
        for b in '01':
            lines.append('%s : %s -> %s' % (nm(w), b, nm((w + b)[1:])))
    return md2mc('\n'.join(lines))

for N in range(1, 5):
    D = nth_last(N)
    print("N=%d : %2d states, minimal %2d  (2^N = %d)"
          % (N, len(D["Q"]), len(min_dfa(D)["Q"]), 2**N))
    assert len(min_dfa(D)["Q"]) >= 2**N

## 4. Animation

All eight windows and the shift edges between them &mdash; every edge drops the oldest bit.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(T3), FuseEdges=True)

## 5. Exercises


1. How many states does "the third-last symbol is a 1" need over a 3-letter alphabet?
2. Why can an NFA do this with 4 states? (Guess now; Chapter 7 confirms.)
3. Is "the third symbol is a 1" also exponential? Why not?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter5/Concept-Exponential-Blow-Up')